In [ ]:
# Training config — injected at submit time by KaggleTrainingAdapter
CONFIG = {{config}}
print('Experiment:', CONFIG['experiment_name'])
print('Model     :', CONFIG['model'])
print('Epochs    :', CONFIG['epochs'])

In [ ]:
import subprocess, sys, re
from pathlib import Path

dataset_slug = CONFIG['experiment_name'] + '-data'
# Kaggle changed mount path: now /kaggle/input/datasets/<owner>/<slug>/ not /kaggle/input/<slug>/
_candidates = list(Path('/kaggle/input').glob(f'**/{dataset_slug}'))
input_base = _candidates[0] if _candidates else Path('/kaggle/input') / dataset_slug
print(f'Dataset mounted at: {input_base}')

# Test with the exact op that fails in training: torch.embedding uses a custom CUDA
# kernel (not cuBLAS), so it's the first op to surface SM-version mismatches.
_CUDA_KERNEL_TEST = '''
import torch, sys
try:
    w = torch.zeros(100, 8).cuda()
    idx = torch.zeros(2, 3, dtype=torch.long).cuda()
    torch.embedding(w, idx)
    cap = torch.cuda.get_device_capability()
    print(f"CUDA OK gpu={torch.cuda.get_device_name(0)} sm={cap[0]}{cap[1]:02d} torch={torch.__version__}")
except Exception as e:
    cap = torch.cuda.get_device_capability() if torch.cuda.is_available() else (0, 0)
    print(f"CUDA FAIL sm={cap[0]}{cap[1]:02d} torch={torch.__version__}: {e}", file=sys.stderr)
    sys.exit(1)
'''

cuda_test = subprocess.run([sys.executable, '-c', _CUDA_KERNEL_TEST], capture_output=True, text=True)
if 'CUDA OK' in cuda_test.stdout:
    print(cuda_test.stdout.strip())
else:
    print("CUDA embedding test failed:", (cuda_test.stderr or cuda_test.stdout).strip()[:400])

    smi = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
    m = re.search(r'CUDA Version:\s*(\d+)\.(\d+)', smi.stdout)
    cuda_ver = int(m.group(1)) * 100 + int(m.group(2)) if m else 0

    sm_test = subprocess.run(
        [sys.executable, '-c',
         'import torch; cap=torch.cuda.get_device_capability(); print(cap[0]*100+cap[1])'],
        capture_output=True, text=True,
    )
    sm_ver = int(sm_test.stdout.strip()) if sm_test.stdout.strip().isdigit() else 0
    print(f"CUDA driver={cuda_ver}  GPU SM={sm_ver}")

    if sm_ver < 700:
        cu_tag = 'cu118'
        torch_spec = 'torch==2.2.2'
        print(f"WARNING: P100/Pascal GPU detected (SM={sm_ver}). Pinning to torch==2.2.2+cu118.")
    elif cuda_ver >= 1208 or sm_ver >= 900:
        cu_tag, torch_spec = 'cu128', 'torch'
    elif cuda_ver >= 1204 or sm_ver >= 800:
        cu_tag, torch_spec = 'cu124', 'torch'
    else:
        cu_tag, torch_spec = 'cu121', 'torch'

    print(f"Reinstalling {torch_spec} from {cu_tag} ...")
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', '--force-reinstall',
        torch_spec, '--index-url', f'https://download.pytorch.org/whl/{cu_tag}'
    ], check=True)

    verify = subprocess.run([sys.executable, '-c', _CUDA_KERNEL_TEST], capture_output=True, text=True)
    result = verify.stdout.strip() or verify.stderr.strip()[:400]
    print("After reinstall:", result)
    if 'CUDA OK' not in verify.stdout:
        raise RuntimeError(f"torch reinstall did not fix CUDA kernel compatibility: {result}")

# Install deps not pre-installed on Kaggle (wheel uses --no-deps so these won't auto-install).
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'bitsandbytes', 'peft'], check=True)
print('bitsandbytes + peft ready.')

# Always force-reinstall the project wheel so local code changes reach Kaggle.
# List what Kaggle actually mounted so we can diagnose missing-file issues.
import os
print(f'Contents of /kaggle/input:')
for root, dirs, files in os.walk('/kaggle/input'):
    level = root.replace('/kaggle/input', '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')
    for f in files:
        print(f'{indent}  {f}')

wheels = list(input_base.glob('*.whl')) or list(input_base.glob('**/*.whl'))
if not wheels:
    raise FileNotFoundError(f'No .whl found under {input_base} — re-run the training trigger to rebuild the dataset.')
wheel = wheels[0]
print(f'Installing {wheel.name} (force-reinstall) …')
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--no-deps', '--force-reinstall', '-q', str(wheel)],
    check=True,
)
print('Done.')

In [ ]:
import shutil
from pathlib import Path

# Kaggle mounts attached datasets at /kaggle/input/datasets/<owner>/<slug>/
dataset_slug = CONFIG['experiment_name'] + '-data'
_candidates = list(Path('/kaggle/input').glob(f'**/{dataset_slug}'))
input_base = _candidates[0] if _candidates else Path('/kaggle/input') / dataset_slug

# Copy flat .jsonl files from the dataset into data/
data_dst = Path('data')
data_dst.mkdir(exist_ok=True)
for jsonl in input_base.glob('*.jsonl'):
    shutil.copy(jsonl, data_dst / jsonl.name)
    print(f'Copied {jsonl.name} ({jsonl.stat().st_size // 1000} KB)')

In [ ]:
import subprocess, sys, os

# bitsandbytes quantised models crash with DataParallel (multi-GPU).
# Pin to GPU 0 so the Trainer never wraps with DataParallel.
env = {**os.environ, 'CUDA_VISIBLE_DEVICES': '0', 'HF_HOME': '/kaggle/working/.hf_cache'}

# Package is installed, so interactors.cli.train is importable as a module directly
cmd = [
    sys.executable, '-m', 'interactors.cli.train',
    '--model', CONFIG['model'],
    '--epochs', str(CONFIG['epochs']),
    '--patience', str(CONFIG['patience']),
    '--warmup-ratio', str(CONFIG['warmup_ratio']),
    '--train-data', 'data/train.jsonl',
    '--eval-data', 'data/eval.jsonl',
    '--output-dir', 'models/checkpoints',
    '--progress-path', '/kaggle/working/progress.json',
]
print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True, env=env)
print('Training complete.')

In [ ]:
import tarfile
from pathlib import Path

checkpoint_dir = Path('models/checkpoints')
output_archive = Path('/kaggle/working/checkpoint.tar.gz')

if not checkpoint_dir.exists():
    raise FileNotFoundError(f'Checkpoint not found at {checkpoint_dir}')

with tarfile.open(output_archive, 'w:gz') as tf:
    tf.add(checkpoint_dir, arcname='checkpoints')

print(f'Packaged -> {output_archive} ({output_archive.stat().st_size / 1e6:.1f} MB)')

import shutil
shutil.rmtree(checkpoint_dir)
print(f'Removed {checkpoint_dir} to free disk space.')